# Euler–Maruyama Convergence

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare OU discretizations at several Δt values against the exact mean.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/em-convergence/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's OU defaults: dX = theta (mu - X) dt + sigma dW, X0 = 2, horizon T = 5
T, theta, mu, sigma, x0 = 5.0, 1.2, 0.0, 0.8, 2.0
n_paths, fine = 20000, 2 ** 12          # fine grid for the reference solution

# Exact OU transition over a step h, driven by the SAME Brownian increments:
#   X_{t+h} = mu + (X_t - mu) e^{-theta h} + sigma ∫ e^{-theta (t+h-s)} dW_s
# On the fine grid the integral is taken exactly enough with tiny steps.
dW_fine = math.sqrt(T / fine) * torch.randn(n_paths, fine, dtype=torch.float64)


def em(n_steps):
    """Euler-Maruyama with n_steps, using sums of the fine increments."""
    h = T / n_steps
    dW = dW_fine.view(n_paths, n_steps, -1).sum(2)
    x = torch.full((n_paths,), x0, dtype=torch.float64)
    for k in range(n_steps):
        x = x + theta * (mu - x) * h + sigma * dW[:, k]
    return x


reference = em(fine)                                  # stands in for the exact pathwise solution
exact_mean = mu + (x0 - mu) * math.exp(-theta * T)

print("  steps     dt      strong E|X_N - X(T)|  ratio    weak |E X_N - E X(T)|  ratio")
prev_s = prev_w = None
for n in (16, 32, 64, 128, 256):
    h = T / n
    strong = (em(n) - reference).abs().mean().item()
    # EM's mean obeys m <- m + theta (mu - m) h exactly, so the weak error needs no sampling
    weak = abs(mu + (x0 - mu) * (1 - theta * h) ** n - exact_mean)
    rs = f"{prev_s / strong:5.2f}" if prev_s else "     "
    rw = f"{prev_w / weak:5.2f}" if prev_w else "     "
    print(f"{n:6d}   {h:.4f}    {strong:.6f}            {rs}    {weak:.6f}               {rw}")
    prev_s, prev_w = strong, weak
print(f"\n(a Monte Carlo estimate of the mean with {n_paths} paths has noise ~ "
      f"{math.sqrt(sigma ** 2 / (2 * theta) / n_paths):.4f}, larger than these weak errors)")
# Both ratios ~2: order 1. Additive noise gives EM strong order 1; with noise that depends
# on X it drops to 1/2. Comparing ONE path with the exact MEAN never converges.

  steps     dt      strong E|X_N - X(T)|  ratio    weak |E X_N - E X(T)|  ratio
    16   0.3125    0.080506                     0.003873                    
    32   0.1562    0.037388             2.15    0.002355                1.64
    64   0.0781    0.018046             2.07    0.001285                1.83
   128   0.0391    0.008798             2.05    0.000670                1.92
   256   0.0195    0.004302             2.04    0.000342                1.96

(a Monte Carlo estimate of the mean with 20000 paths has noise ~ 0.0037, larger than these weak errors)
